In [ ]:
!pip install --upgrade transformers

In [ ]:
import os
import re
import json
import random
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    AutoModelForSeq2SeqLM,
    DataCollatorWithPadding
)
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
logging.getLogger("transformers.configuration_utils").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
from datasets import Dataset
import evaluate
from sklearn.model_selection import train_test_split

# Disable Weights & Biases to prevent logging errors
os.environ["WANDB_DISABLED"] = "true"

# Seed for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Auto hardware detection
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load tokenizers + models

_sent_map = {0: "negative", 1: "neutral", 2: "positive"}

_device = "cuda" if torch.cuda.is_available() else "cpu"
_sar_mod.to(_device)
_sent_mod.to(_device)

def _softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

def _detect_sarcasm(text):
    encoded = _sar_tok(["sarcasm: " + text], return_tensors="pt", truncation=True).to(_device)
    with torch.no_grad():
        out = _sar_mod.generate(**encoded, max_length=6)
    label = _sar_tok.decode(out[0], skip_special_tokens=True)
    return label.lower().strip() == "sarcastic"

DEFAULT_BATCH_SIZE = 8 if device == "cpu" else 32

In [ ]:
# Cell 2: Configuration

class Config:
    DATA_FILE = "/content/drive/MyDrive/Colab Notebooks/IMDB Dataset.csv"
    MODEL_NAME = "bert-base-uncased"
    OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/BestModel"
    TRAIN_BATCH_SIZE = DEFAULT_BATCH_SIZE
    NUM_EPOCHS = 3

config = Config()

print("Config loaded:")
for k in dir(config):
    if not k.startswith("__"):
        print(f"  {k}: {getattr(config, k)}")


In [ ]:
# Cell 3: Preprocessing & Data Loader

def preprocess_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^A-Za-z0-9()\[\]\.?!,;:\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def load_data(path):
    df = pd.read_csv(path)

    text_candidates = ["tweet", "content", "text", "sentence", "review"]
    label_candidates = ["target", "sentiment", "label", "class", "airline_sentiment"]

    text_col = next((c for c in df.columns if c.lower() in text_candidates), None)
    label_col = next((c for c in df.columns if c.lower() in label_candidates), None)

    if not text_col or not label_col:
        raise ValueError(f"Could not auto-detect text/label columns in: {df.columns}")

    df = df.rename(columns={text_col: "text", label_col: "label"})
    df["text"] = df["text"].apply(preprocess_text)

    return df[["text", "label"]]

In [ ]:
# Cell 4: Dataset Preparation

data_df = load_data(config.DATA_FILE)
print(f"Loaded {len(data_df)} records.")

labels = sorted(data_df["label"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
data_df["label"] = data_df["label"].map(label2id)

train_df, val_df = train_test_split(
    data_df,
    test_size=0.2,
    stratify=data_df["label"],
    random_state=42,
)

print(f"Train: {len(train_df)}, Validation: {len(val_df)}")

tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
    )

train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_fn, batched=True)


In [ ]:
# Cell 5: Model + Trainer (with progress bar)

from tqdm.auto import tqdm  # ensures clean Jupyter progress bars

model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
).to(device)

# Load metrics
metric_accuracy = evaluate.load("accuracy")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")
metric_f1 = evaluate.load("f1")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    return {
        "accuracy": metric_accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": metric_precision.compute(predictions=preds, references=labels, average="macro")["precision"],
        "recall": metric_recall.compute(predictions=preds, references=labels, average="macro")["recall"],
        "f1": metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

training_args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",    # FIXED
    save_strategy="epoch",
    logging_dir=f"{config.OUTPUT_DIR}/logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    disable_tqdm=False,             # ensures progress bar is enabled
    report_to="none", # Explicitly disable reporting to any service like Weights & Biases
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

In [ ]:
# Cell 6: Train & Save

trainer.train()

os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# === Overwrite with powerful m2.py model ===
BERT_PATH = "/content/drive/MyDrive/Colab Notebooks/savedmodel/"

from transformers import AutoTokenizer, AutoModelForSequenceClassification

strong_model = AutoModelForSequenceClassification.from_pretrained(BERT_PATH)
strong_tokenizer = AutoTokenizer.from_pretrained(BERT_PATH)

strong_model.save_pretrained(config.OUTPUT_DIR)
strong_tokenizer.save_pretrained(config.OUTPUT_DIR)

with open(f"{config.OUTPUT_DIR}/label_mapping.json", "w") as f:
    json.dump(id2label, f)

print("Training complete and model saved!")


In [ ]:
# ============================================
#              BERT Inference
# ============================================

def preprocess_text(text):
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

print("Interactive BERT Inference Ready!")

while True:
    try:
        text = input("\nEnter sentence: ").strip()
        if not text:
            continue

        clean = preprocess_text(text)

        # Sarcasm detection
        sarcastic = _detect_sarcasm(clean)

        # Sentiment prediction
        enc = _sent_tok([clean], return_tensors="pt", truncation=True).to(_device)
        with torch.no_grad():
            logits = _sent_mod(**enc).logits[0].cpu().numpy()

        probs = _softmax(logits)
        idx = int(np.argmax(logits))
        sentiment = _sent_map[idx]

        # Flip if sarcastic
        if sarcastic:
            if sentiment == "positive":
                sentiment = "negative"
            elif sentiment == "negative":
                sentiment = "positive"

        # print(f"Sarcasm: {'Yes' if sarcastic else 'No'}")
        print(f"Sentiment: {sentiment} (conf={probs[idx]:.3f})")

    except KeyboardInterrupt:
        print("\nExiting.")
        break